In [10]:
import pandas as pd
import numpy as np
import sqlite3

In [11]:
df = pd.read_csv('ecommerce_cleaned.csv')

print("=== LOADING CLEANED DATA ===")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 3 rows:")
print(df.head(3))

=== LOADING CLEANED DATA ===
Shape: (34500, 23)
Columns: ['order_id', 'customer_id', 'product_id', 'category', 'price', 'discount', 'quantity', 'payment_method', 'order_date', 'delivery_time_days', 'region', 'returned', 'total_amount', 'shipping_cost', 'profit_margin', 'customer_age', 'customer_gender', 'year', 'month', 'day', 'weekday', 'quarter', 'unit_price']

First 3 rows:
  order_id customer_id product_id     category   price  discount  quantity  \
0  O100000      C17270    P234890         Home  164.08      0.15         1   
1  O100001      C17603    P228204      Grocery   24.73      0.00         1   
2  O100002      C10860    P213892  Electronics  175.58      0.05         1   

  payment_method  order_date  delivery_time_days  ... shipping_cost  \
0    Credit Card  2023-12-23                   4  ...          7.88   
1    Credit Card  2025-04-03                   6  ...          4.60   
2    Credit Card  2024-10-08                   4  ...          6.58   

  profit_margin  custo

In [13]:
conn = sqlite3.connect('ecommerce.db')
df.to_sql('sales', conn, if_exists='replace', index=False)

print("\nData loaded into SQLite database: ecommerce.db")
print(f"Table name: sales")
print(f"Total rows: {df.shape[0]}")


Data loaded into SQLite database: ecommerce.db
Table name: sales
Total rows: 34500


<u>**QUERY 1: Monthly Sales Trend**</u>

**Purpose**: Shows total sales and number of orders for each month to identify seasonal patterns.

In [14]:
query1 = """
SELECT 
    year,
    month,
    SUM(total_amount) as total_sales,
    COUNT(DISTINCT order_id) as order_count
FROM sales
GROUP BY year, month
ORDER BY year, month;
"""
df1 = pd.read_sql_query(query1, conn)
print("QUERY 1: Monthly Sales Trend")
print(df1.to_string(index=False))
df1.to_csv('sql_results_monthly_trend.csv', index=False)

QUERY 1: Monthly Sales Trend
 year  month  total_sales  order_count
 2023      9    151135.60          876
 2023     10    262502.74         1468
 2023     11    240286.91         1385
 2023     12    255617.03         1499
 2024      1    217766.09         1386
 2024      2    228013.98         1358
 2024      3    248176.28         1423
 2024      4    265596.66         1436
 2024      5    264875.23         1427
 2024      6    234155.38         1381
 2024      7    253369.40         1501
 2024      8    212613.20         1378
 2024      9    218873.90         1418
 2024     10    258931.27         1482
 2024     11    250572.04         1474
 2024     12    278154.19         1469
 2025      1    224390.21         1442
 2025      2    215338.19         1337
 2025      3    239346.81         1481
 2025      4    263024.03         1431
 2025      5    248527.46         1494
 2025      6    248776.28         1430
 2025      7    249593.50         1471
 2025      8    243808.50         1

<u>**QUERY 2: Sales by Category**</u>

**Purpose**: Identifies which product categories generate the most revenue and sales volume.

In [16]:
query2 = """
SELECT 
    category,
    SUM(total_amount) as total_revenue,
    SUM(quantity) as total_quantity,
    COUNT(DISTINCT order_id) as order_count
FROM sales
GROUP BY category
ORDER BY total_revenue DESC;
"""
df2 = pd.read_sql_query(query2, conn)
print("QUERY 2: Sales by Category")
print(df2.to_string(index=False))
df2.to_csv('sql_results_category_sales.csv', index=False)

QUERY 2: Sales by Category
   category  total_revenue  total_quantity  order_count
Electronics     3319206.50            9343         6180
       Home     1077681.52            8229         5487
     Sports      629825.54            6244         4171
    Fashion      471545.80            9148         6254
     Beauty      153019.38            6122         4103
       Toys      132013.80            6313         4247
    Grocery       82000.51            6031         4058


<u>**QUERY 3: Sales by Weekday**</u>

**Purpose**: Shows which days of the week have highest sales to optimize marketing and staffing.

In [18]:
query3 = """
SELECT 
    weekday,
    SUM(total_amount) as total_revenue,
    COUNT(DISTINCT order_id) as order_count,
    AVG(total_amount) as avg_order_value
FROM sales
GROUP BY weekday
ORDER BY total_revenue DESC;
"""
df3 = pd.read_sql_query(query3, conn)
print("QUERY 3: Sales by Weekday")
print(df3.to_string(index=False))
df3.to_csv('sql_results_weekday_sales.csv', index=False)

QUERY 3: Sales by Weekday
  weekday  total_revenue  order_count  avg_order_value
  Tuesday      866319.23         4971       174.274639
   Monday      853282.82         4924       173.290581
   Sunday      849144.79         4924       172.450201
Wednesday      834664.71         4939       168.994677
   Friday      826720.42         4881       169.375214
 Thursday      824795.02         5058       163.067422
 Saturday      810366.06         4803       168.720812


<u>**QUERY 4: Sales by Payment Method**</u> 

**Purpose**: Shows customer payment preferences to optimize checkout options.


In [19]:
query4 = """
SELECT 
    payment_method,
    SUM(total_amount) as total_revenue,
    COUNT(DISTINCT order_id) as order_count
FROM sales
GROUP BY payment_method
ORDER BY total_revenue DESC;
"""
df4 = pd.read_sql_query(query4, conn)
print("QUERY 4: Sales by Payment Method")
print(df4.to_string(index=False))
df4.to_csv('sql_results_payment_method.csv', index=False)

QUERY 4: Sales by Payment Method
payment_method  total_revenue  order_count
   Credit Card     2056787.40        12170
    Debit Card     1460210.97         8505
           COD      715571.90         4160
           UPI      713642.96         4156
        PayPal      576523.32         3444
        Wallet      342556.50         2065


<u>**QUERY 5: Weekday vs Weekend Sales**</u> 

**Purpose**: Compares weekday vs weekend performance to identify shopping behavior patterns.

In [22]:
query5 = """
SELECT 
    CASE 
        WHEN weekday IN ('Saturday', 'Sunday') THEN 'Weekend'
        ELSE 'Weekday'
    END as day_type,
    SUM(total_amount) as total_revenue,
    COUNT(DISTINCT order_id) as order_count,
    AVG(total_amount) as avg_order_value
FROM sales
GROUP BY day_type;
"""
df5 = pd.read_sql_query(query5, conn)
print("QUERY 5: Weekday vs Weekend Sales")
print(df5.to_string(index=False))
df5.to_csv('sql_results_weekday_vs_weekend.csv', index=False)

QUERY 5: Weekday vs Weekend Sales
day_type  total_revenue  order_count  avg_order_value
 Weekday     4205782.20        24773       169.772825
 Weekend     1659510.85         9727       170.608703


<u>**QUERY 6: Sales by Customer Gender**</u> 

**Purpose**: Analyzes spending patterns between male and female customers.

In [23]:
query6 = """
SELECT 
    customer_gender,
    COUNT(DISTINCT customer_id) as customer_count,
    SUM(total_amount) as total_revenue,
    AVG(total_amount) as avg_order_value
FROM sales
GROUP BY customer_gender;
"""
df6 = pd.read_sql_query(query6, conn)
print(df6.to_string(index=False))
df6.to_csv('sql_results_customer_gender.csv', index=False)

customer_gender  customer_count  total_revenue  avg_order_value
         Female            7026     2853296.96       169.657329
           Male            6981     2793055.33       171.248028
          Other            1276      218940.76       159.577813


<u>**QUERY 7: Month-over-Month Revenue Growth**</u> 

**Purpose**: Calculates month-over-month growth percentage to track business performance.

query7 = """
WITH monthly AS (
    SELECT 
        year,
        month,
        SUM(total_amount) as revenue
    FROM sales
    GROUP BY year, month
),
growth AS (
    SELECT 
        year,
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY year, month) as prev_month_revenue,
        ROUND(((revenue - LAG(revenue) OVER (ORDER BY year, month)) / 
               LAG(revenue) OVER (ORDER BY year, month)) * 100, 2) as growth_pct
    FROM monthly
)
SELECT * FROM growth;
"""
df7 = pd.read_sql_query(query7, conn)
print("QUERY 7: Month-over-Month Revenue Growth")
print(df7.to_string(index=False))
df7.to_csv('sql_results_growth.csv', index=False)

<u>**QUERY 8: Top 10 Customers by Spending**</u> 

**Purpose**: Identifies highest-value customers for loyalty programs and targeted marketing.

In [28]:
query8 = """
SELECT 
    customer_id,
    COUNT(DISTINCT order_id) as order_count,
    SUM(total_amount) as total_spent,
    AVG(total_amount) as avg_order_value
FROM sales
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;
"""
df8 = pd.read_sql_query(query8, conn)
print("QUERY 8: Top 10 Customers by Spending")
print(df8.to_string(index=False))
df8.to_csv('sql_results_top_customers.csv', index=False)

QUERY 8: Top 10 Customers by Spending
customer_id  order_count  total_spent  avg_order_value
     C16655           10     13885.10      1388.510000
     C13565            5     11984.28      2396.856000
     C15379            3     11375.58      3791.860000
     C17116            5      7424.34      1484.868000
     C15644            7      7244.07      1034.867143
     C10975           10      7209.12       720.912000
     C16382           10      6973.06       697.306000
     C13767            6      6546.60      1091.100000
     C15254            5      6509.95      1301.990000
     C12427            5      6480.16      1296.032000


<u>**SUMMARY**</u> 

In [31]:
conn.close()
print("=== PROJECT SUMMARY ===")

print("1. ecommerce_cleaned.csv - Cleaned dataset")
print("2. ecommerce.db - SQLite database")
print("3. sql_results_monthly_trend.csv - Monthly sales pattern")
print("4. sql_results_category_sales.csv - Category performance")
print("5. sql_results_weekday_sales.csv - Daily sales pattern")
print("6. sql_results_payment_method.csv - Payment preferences")
print("7. sql_results_weekday_vs_weekend.csv - Weekend comparison")
print("8. sql_results_customer_gender.csv - Gender analysis")
print("9. sql_results_growth.csv - Monthly growth rate")
print("10. sql_results_top_customers.csv - Top customers")

=== PROJECT SUMMARY ===
1. ecommerce_cleaned.csv - Cleaned dataset
2. ecommerce.db - SQLite database
3. sql_results_monthly_trend.csv - Monthly sales pattern
4. sql_results_category_sales.csv - Category performance
5. sql_results_weekday_sales.csv - Daily sales pattern
6. sql_results_payment_method.csv - Payment preferences
7. sql_results_weekday_vs_weekend.csv - Weekend comparison
8. sql_results_customer_gender.csv - Gender analysis
9. sql_results_growth.csv - Monthly growth rate
10. sql_results_top_customers.csv - Top customers
